In [ ]:
# 讀入套件
import numpy as np
import pandas as pd
from collections import Counter
from typing import Optional, Union, Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

# 用於輸出 Excel 檔案
from openpyxl import Workbook
from openpyxl.styles import PatternFill

# --- 資料預處理 (與原始程式碼相同) ---
class AdultDataPreprocessor:
    def __init__(self, train_path: str, test_path: str):
        self.train_path = train_path
        self.test_path = test_path
        self.column_names = [
            'age', 'workclass', 'fnlwgt', 'education', 'education-num',
            'marital-status', 'occupation', 'relationship', 'race', 'sex',
            'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
        ]
        self.categorical_features = [
            'workclass', 'education', 'marital-status', 'occupation',
            'relationship', 'race', 'sex', 'native-country'
        ]
        self.numerical_features = [
            'age', 'fnlwgt', 'education-num', 'capital-gain',
            'capital-loss', 'hours-per-week'
        ]
    
    def load_and_preprocess(self, filepath: str, is_test: bool = False) -> pd.DataFrame:
        df = pd.read_csv(filepath, names=self.column_names, 
                         skipinitialspace=True, na_values='?')
        if is_test:
            df = df[~df['age'].astype(str).str.contains('\\|', na=False)].reset_index(drop=True)
            df['income'] = df['income'].str.rstrip('.')
        
        initial_rows = len(df)
        df = df.drop_duplicates()
        removed_duplicates = initial_rows - len(df)
        if removed_duplicates > 0:
            print(f"移除 {removed_duplicates} 筆重複資料")
        
        for col in self.categorical_features:
            if df[col].isnull().any():
                mode_value = df[col].mode()[0]
                df[col].fillna(mode_value, inplace=True)
                print(f"欄位 {col} 有缺失值，使用眾數 '{mode_value}' 填補")
        
        for col in self.numerical_features:
            if df[col].isnull().any():
                median_value = df[col].median()
                df[col].fillna(median_value, inplace=True)
                print(f"欄位 {col} 有缺失值，使用中位數 {median_value} 填補")
            df[col] = pd.to_numeric(df[col], errors='coerce')
            if df[col].isnull().any():
                df[col].fillna(df[col].median(), inplace=True)
        
        df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})
        return df
    
    def preprocess_train(self) -> pd.DataFrame:
        print("=" * 50)
        print("處理訓練資料...")
        print("=" * 50)
        train_df = self.load_and_preprocess(self.train_path, is_test=False)
        print(f"訓練資料筆數: {len(train_df)}")
        print(f"特徵數量: {len(train_df.columns) - 1}")
        return train_df
    
    def preprocess_test(self) -> pd.DataFrame:
        print("\n" + "=" * 50)
        print("處理測試資料...")
        print("=" * 50)
        test_df = self.load_and_preprocess(self.test_path, is_test=True)
        print(f"測試資料筆數: {len(test_df)}")
        return test_df

# --- C5.0 弱學習器 (單一決策樹) ---
class C50Node:
    def __init__(self):
        self.feature = None
        self.threshold = None
        self.categories = None
        self.is_leaf = False
        self.prediction = None
        self.children = {}
        self.info_gain = 0.0
        self.samples = 0
        self.class_distribution = {}

class C50DecisionTree:
    def __init__(self, max_depth: Optional[int] = None, min_samples_split: int = 2,
                 min_samples_leaf: int = 1, max_nodes: Optional[int] = None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_nodes = max_nodes
        self.root = None
        self.node_count = 0
        self.feature_names = None
        self.categorical_features = []
        self.numerical_features = []
    
    def _entropy(self, y: np.ndarray, sample_weight: np.ndarray) -> float:
        if len(y) == 0 or np.sum(sample_weight) == 0:
            return 0.0
        
        weighted_counts = {}
        for label in np.unique(y):
            weighted_counts[label] = np.sum(sample_weight[y == label])
            
        total_weight = np.sum(list(weighted_counts.values()))
        if total_weight == 0: return 0.0
        
        probabilities = np.array(list(weighted_counts.values())) / total_weight
        return -np.sum(probabilities * np.log2(probabilities + 1e-10))

    def _gain_ratio(self, y: np.ndarray, splits: List[Tuple[np.ndarray, np.ndarray]], 
                    parent_weights: np.ndarray) -> float:
        parent_entropy = self._entropy(y, parent_weights)
        total_weight = np.sum(parent_weights)
        
        weighted_entropy = 0.0
        split_info = 0.0
        
        for split_y, split_weights in splits:
            if len(split_y) > 0 and np.sum(split_weights) > 0:
                weight = np.sum(split_weights) / total_weight
                weighted_entropy += weight * self._entropy(split_y, split_weights)
                split_info -= weight * np.log2(weight + 1e-10)
        
        info_gain = parent_entropy - weighted_entropy
        return info_gain / split_info if split_info > 0 else 0.0
    
    def _best_split(self, X: pd.DataFrame, y: np.ndarray, sample_weight: np.ndarray) -> Tuple[str, Union[float, List], float]:
        best_gain_ratio = -1
        best_feature = None
        best_split_value = None
        
        for feature in X.columns:
            gain_ratio, split_value = 0, None
            values = X[feature].unique()
            
            if feature in self.categorical_features and len(values) > 1:
                splits = []
                for cat in values:
                    mask = X[feature] == cat
                    if np.any(mask):
                        splits.append((y[mask], sample_weight[mask]))
                if all(len(s[0]) >= self.min_samples_leaf for s in splits):
                    gain_ratio = self._gain_ratio(y, splits, sample_weight)
                    split_value = list(values)

            elif feature in self.numerical_features and len(values) > 1:
                best_threshold_gain = -1
                best_threshold = None
                
                sorted_values = np.sort(values)
                if len(sorted_values) > 50:
                    candidates = np.percentile(sorted_values, np.linspace(5, 95, 20))
                else:
                    candidates = (sorted_values[:-1] + sorted_values[1:]) / 2
                
                for threshold in candidates:
                    left_mask = X[feature] <= threshold
                    right_mask = ~left_mask
                    if np.sum(left_mask) >= self.min_samples_leaf and np.sum(right_mask) >= self.min_samples_leaf:
                        splits = [(y[left_mask], sample_weight[left_mask]), (y[right_mask], sample_weight[right_mask])]
                        current_gain = self._gain_ratio(y, splits, sample_weight)
                        if current_gain > best_threshold_gain:
                            best_threshold_gain = current_gain
                            best_threshold = threshold
                
                gain_ratio = best_threshold_gain
                split_value = best_threshold

            if gain_ratio > best_gain_ratio:
                best_gain_ratio = gain_ratio
                best_feature = feature
                best_split_value = split_value
                
        return best_feature, best_split_value, best_gain_ratio

    def _build_tree(self, X: pd.DataFrame, y: np.ndarray, sample_weight: np.ndarray, depth: int = 0) -> C50Node:
        node = C50Node()
        node.samples = len(y)
        node.class_distribution = dict(Counter(y))
        self.node_count += 1
        
        weighted_counts = {label: np.sum(sample_weight[y == label]) for label in np.unique(y)}
        most_common = max(weighted_counts, key=weighted_counts.get) if weighted_counts else Counter(y).most_common(1)[0][0]

        if (len(np.unique(y)) == 1 or
            (self.max_depth is not None and depth >= self.max_depth) or
            len(y) < self.min_samples_split or
            (self.max_nodes is not None and self.node_count >= self.max_nodes)):
            node.is_leaf = True
            node.prediction = most_common
            return node

        best_feature, best_split_value, best_gain_ratio = self._best_split(X, y, sample_weight)

        if best_feature is None or best_gain_ratio <= 0:
            node.is_leaf = True
            node.prediction = most_common
            return node
        
        node.feature = best_feature
        node.info_gain = best_gain_ratio

        if best_feature in self.categorical_features:
            node.categories = best_split_value
            for category in best_split_value:
                mask = X[best_feature] == category
                if mask.sum() > 0:
                    node.children[category] = self._build_tree(X[mask], y[mask], sample_weight[mask], depth + 1)
        else:
            node.threshold = best_split_value
            left_mask = X[best_feature] <= best_split_value
            if left_mask.sum() > 0:
                node.children['left'] = self._build_tree(X[left_mask], y[left_mask], sample_weight[left_mask], depth + 1)
            if (~left_mask).sum() > 0:
                node.children['right'] = self._build_tree(X[~left_mask], y[~left_mask], sample_weight[~left_mask], depth + 1)
        
        return node
    
    def fit(self, X: pd.DataFrame, y: np.ndarray, 
            categorical_features: List[str], numerical_features: List[str],
            sample_weight: Optional[np.ndarray] = None):
        self.feature_names = list(X.columns)
        self.categorical_features = categorical_features
        self.numerical_features = numerical_features
        self.node_count = 0
        
        if sample_weight is None:
            sample_weight = np.ones(len(y)) / len(y)
            
        self.root = self._build_tree(X, y, sample_weight)
    
    def _predict_sample(self, x: pd.Series, node: C50Node) -> int:
        if node.is_leaf:
            return node.prediction
        
        feature_value = x[node.feature]
        
        if node.feature in self.categorical_features:
            if feature_value in node.children:
                return self._predict_sample(x, node.children[feature_value])
            return max(node.class_distribution, key=node.class_distribution.get)
        else:
            try:
                child_node_key = 'left' if float(feature_value) <= float(node.threshold) else 'right'
                if child_node_key in node.children:
                    return self._predict_sample(x, node.children[child_node_key])
            except (TypeError, ValueError):
                pass
            return max(node.class_distribution, key=node.class_distribution.get)
    
    def predict(self, X: pd.DataFrame) -> np.ndarray:
        if self.root is None:
            raise ValueError("模型尚未訓練")
        return np.array([self._predict_sample(row, self.root) for _, row in X.iterrows()])

# +++ C5.0 Boosting 分類器 +++
class C50Boosted:
    def __init__(self, n_estimators: int = 10, **tree_params):
        self.n_estimators = n_estimators
        self.tree_params = tree_params
        self.trees = []
        self.alphas = []        

    def fit(self, X: pd.DataFrame, y: np.ndarray, 
            categorical_features: List[str], numerical_features: List[str]):
        
        n_samples = len(y)
        sample_weight = np.full(n_samples, 1 / n_samples)
        
        print(f"\n開始訓練 C5.0 Boosting 模型 (共 {self.n_estimators} 個)...")

        for i in range(self.n_estimators):
            tree = C50DecisionTree(**self.tree_params)
            tree.fit(X, y, categorical_features, numerical_features, sample_weight)
            
            y_pred = tree.predict(X)
            misclassified = y != y_pred
            error = np.sum(sample_weight[misclassified]) / np.sum(sample_weight)
            error = np.clip(error, 1e-10, 1 - 1e-10)
            alpha = 0.5 * np.log((1 - error) / error)
            weights_update = np.exp(-alpha * np.where(misclassified, -1, 1))
            sample_weight *= weights_update
            sample_weight /= np.sum(sample_weight)
            
            self.trees.append(tree)
            self.alphas.append(alpha)

            #找參數時可註解
            #print(f"  估計器 {i+1}/{self.n_estimators} 完成, 錯誤率: {error:.4f}, Alpha: {alpha:.4f}")
            
            if error <= 1e-10:
                print("  偵測到完美分類器，提前停止訓練。")
                break
        
        print("訓練完成")

        # 從 self.trees 列表中收集所有樹的 node_count
        """all_node_counts = [tree.node_count for tree in self.trees]
        
        if all_node_counts:
            total_nodes = sum(all_node_counts)
            avg_nodes = total_nodes / len(all_node_counts)
            
            print(f"  - 總共訓練了 {len(all_node_counts)} 棵樹。")
            print(f"  - 總節點數 (所有樹相加): {total_nodes}")
            print(f"  - 平均節點數 (每棵樹): {avg_nodes:.2f}")
            
            print(f"  - 每棵樹的節點數列表: {all_node_counts}")"""

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        n_samples = len(X)
        class_votes = np.zeros((n_samples, 2))

        for alpha, tree in zip(self.alphas, self.trees):
            predictions = tree.predict(X)
            for i, pred in enumerate(predictions):
                class_votes[i, pred] += alpha
        
        return np.argmax(class_votes, axis=1)

# --- 主程式與評估 ---
def evaluate_model(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1_score': f1,
            'true_positive': tp, 'true_negative': tn, 'false_positive': fp, 'false_negative': fn}

def export_to_excel(y_true: np.ndarray, y_pred: np.ndarray, filename: str = 'predictions.xlsx'):
    wb = Workbook()
    ws = wb.active
    ws.title = "Prediction Results"
    ws['A1'], ws['B1'] = '實際類別', '預測類別'
    ws['A1'].fill = ws['B1'].fill = PatternFill(start_color="FFD700", end_color="FFD700", fill_type="solid")
    green_fill = PatternFill(start_color="90EE90", end_color="90EE90", fill_type="solid")
    red_fill = PatternFill(start_color="FFB6C6", end_color="FFB6C6", fill_type="solid")
    label_map = {0: '<=50K', 1: '>50K'}
    
    for i, (true_val, pred_val) in enumerate(zip(y_true, y_pred), start=2):
        ws[f'A{i}'] = label_map[true_val]
        ws[f'B{i}'] = label_map[pred_val]
        fill = green_fill if true_val == pred_val else red_fill
        ws[f'A{i}'].fill = fill
        ws[f'B{i}'].fill = fill
    
    wb.save(filename)
    print(f"\n預測結果已匯出至: {filename}")

def print_metrics(metrics, title=""):
    print("\n" + "=" * 50)
    if title: print(title.center(50))
    print("=" * 50)
    print(f"  {'準確率 (Accuracy)':<25}: {metrics['accuracy'] * 100:.2f}%")
    print(f"  {'精確度 (Precision)':<25}: {metrics['precision'] * 100:.2f}%")
    print(f"  {'召回率 (Recall)':<25}: {metrics['recall'] * 100:.2f}%")
    print(f"  {'F1-Score':<25}: {metrics['f1_score'] * 100:.2f}%")
    print("\n--- Confusion Matrix ---".center(50))
    print(f"  {'真陽性 (True Positive, TP)':<25}: {metrics['true_positive']:<10}")
    print(f"  {'真陰性 (True Negative, TN)':<25}: {metrics['true_negative']:<10}")
    print(f"  {'偽陽性 (False Positive, FP)':<25}: {metrics['false_positive']:<10}")
    print(f"  {'偽陰性 (False Negative, FN)':<25}: {metrics['false_negative']:<10}")

def main():    
    TRAIN_PATH = 'adult/adult.data'
    TEST_PATH = 'adult/adult.test'
    
    preprocessor = AdultDataPreprocessor(TRAIN_PATH, TEST_PATH)
    train_df = preprocessor.preprocess_train()
    test_df = preprocessor.preprocess_test()
    
    X_train = train_df.drop('income', axis=1)
    y_train = train_df['income'].values
    X_test = test_df.drop('income', axis=1)
    y_test = test_df['income'].values
    
    print("\n" + "=" * 50)
    print("資料集資訊")
    print("=" * 50)
    print(f"訓練集樣本數: {len(X_train)}")
    print(f"測試集樣本數: {len(X_test)}")
    print(f"特徵數量: {X_train.shape[1]}")
    
    train_dist = {int(k): v for k, v in Counter(y_train).items()}
    test_dist = {int(k): v for k, v in Counter(y_test).items()}
    print(f"類別分布 (訓練集): {train_dist}")
    print(f"類別分布 (測試集): {test_dist}")

    # 設定傳給每棵樹的參數 (預剪枝)
    tree_params = {
        'max_depth': 15,
        'min_samples_split': 300,
        'min_samples_leaf': 50,
    }
    
    # 使用 Boosting 分類器
    model = C50Boosted(n_estimators=20, **tree_params)
    
    model.fit(
        X_train, y_train,
        categorical_features=preprocessor.categorical_features,
        numerical_features=preprocessor.numerical_features
    )
    
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    train_metrics = evaluate_model(y_train, y_pred_train)
    print_metrics(train_metrics, "訓練資料性能指標 (Boosting)")
    
    test_metrics = evaluate_model(y_test, y_pred_test)
    print_metrics(test_metrics, "測試資料性能指標 (Boosting)")
    
    export_to_excel(y_test, y_pred_test, 'adult_predictions_C50.xlsx')

    return model, X_train, y_train, preprocessor, X_test, y_test

if __name__ == "__main__":
    model , X_train , y_train, preprocessor , X_test , y_test = main()

1.簡單的樹
'max_depth': 3,
'min_samples_split': 2000,
'min_samples_leaf': 1000

訓練資料準確率 (Accuracy) : 81.53%
測試資料準確率 (Accuracy) : 81.25%

2.放縱的樹
'max_depth': None
'min_samples_split': 2
'min_samples_leaf': 1

訓練資料準確率 (Accuracy) : 100.00%  
測試資料準確率 (Accuracy) : 82.03%

3.追求最好的樹(合理)
'max_depth': 15,
'min_samples_split': 300,
'min_samples_leaf': 50,
boosting:20

訓練資料準確率 (Accuracy) : 86.74%
測試資料準確率 (Accuracy) : 86.40%

使用 graphviz 視覺化 C5.0 決策樹

In [ ]:
from graphviz import Digraph
from IPython.display import display, SVG # 引入 SVG 顯示功能
import os

def visualize_tree(root_node: C50Node, 
                   feature_names: List[str], 
                   filename: str = 'c50_tree',
                   format: str = 'svg', # 預設使用 svg
                   dpi: int = 300):
    """
    :param root_node: 樹的根節點。
    :param feature_names: 特徵名稱列表。
    :param filename: 輸出的檔名。
    :param format: 輸出格式 ('svg', 'png', etc.)。
    :param dpi: 圖片解析度 (僅對 PNG 等點陣圖有效)。
    """
    
    # 1. 不在 Digraph 中指定 format，讓 render 決定
    dot = Digraph(comment='C5.0 Decision Tree')
    dot.attr('node', shape='box', style='rounded,filled', fillcolor='lightblue')
    dot.attr('edge', fontname='helvetica', fontsize='10')
    dot.attr(rankdir='TB', size='15,15')
    
    # 2. 真正使用 dpi 參數
    if format != 'svg':
        dot.attr(dpi=str(dpi))
    
    # --- 內部的 add_nodes_edges 函式 ---
    def add_nodes_edges(node: C50Node, dot: Digraph, parent_name: Optional[str] = None, edge_label: str = ''):
        node_name = str(id(node))
        if node.is_leaf:
            class_dist_str = '\\n'.join([f'Class {k}: {v}' for k, v in node.class_distribution.items()])
            label = (f"Prediction: {node.prediction}\\n"
                     f"Samples: {node.samples}\\n"
                     f"{class_dist_str}")
            dot.node(name=node_name, label=label, shape='ellipse', fillcolor='lightgreen')
        else:
            feature_idx = feature_names.index(node.feature)
            label = (f"Feature: {node.feature}\\n"
                     f"Gain Ratio: {node.info_gain:.4f}\\n"
                     f"Samples: {node.samples}")
            dot.node(name=node_name, label=label, fillcolor='orange')
        if parent_name:
            dot.edge(parent_name, node_name, label=edge_label)
        if not node.is_leaf:
            if node.threshold is not None:
                if 'left' in node.children:
                    edge_lbl = f"<= {node.threshold:.2f}"
                    add_nodes_edges(node.children['left'], dot, node_name, edge_lbl)
                if 'right' in node.children:
                    edge_lbl = f"> {node.threshold:.2f}"
                    add_nodes_edges(node.children['right'], dot, node_name, edge_lbl)
            elif node.categories is not None:
                for category, child_node in node.children.items():
                    add_nodes_edges(child_node, dot, node_name, str(category))

    # 從根節點開始建立圖形
    add_nodes_edges(root_node, dot)

    # --- 修正後的渲染和顯示部分 ---
    try:
        # 3. 讓 render 函式使用 format 參數
        # cleanup=True 會刪除原始的 .gv 檔案
        dot.render(filename, format=format, view=False, cleanup=True)
        
        output_filename = f"{filename}.{format}"
        
        # 4. 顯示正確的儲存訊息
        print(f"決策樹圖已成功儲存至 '{output_filename}'")
        
        # 5. 根據格式在 Jupyter 中正確顯示
        if format == 'svg':
            # 確保 SVG 檔案存在
            if os.path.exists(output_filename):
                display(SVG(output_filename))
            else:
                print(f"錯誤：無法找到儲存的 SVG 檔案: {output_filename}")
        else:
            # 對於 PNG 等，直接 display(dot) 即可
            display(dot)
            
    except Exception as e:
        print(f"繪圖失敗，請確認 Graphviz 已正確安裝並設定好環境變數 PATH。")
        print(f"錯誤訊息: {e}")

In [ ]:
# 準備繪圖所需的變數
if 'model' in locals() and 'X_train' in locals():
    # model.trees 是一個列表，[0] 代表第一棵樹
    first_tree = model.trees[0] 
    
    # 取得特徵名稱
    feature_names_list = X_train.columns.tolist()
    
    print("繪圖變數 'first_tree' 和 'feature_names_list' 已準備就緒。")
else:
    print("錯誤：找不到 'model' 或 'X_train'。請先完整執行 Cell 1 (main 函式)。")

In [ ]:
# Cell 5 (修改後)

# --- 範例 1：產生 SVG 向量圖 (推薦) ---
# 檔名會是 C5.0_Tree_Vector.svg
if 'first_tree' in locals():
    visualize_tree(first_tree.root, 
                   feature_names_list, 
                   filename='C5.0_Tree_Vector', 
                   format='svg')
else:
    print("請先執行上一個 Cell 來準備變數。")

# --- 範例 2：產生高解析度 PNG ---
#檔名會是 C5.0_Tree_High_Res.png
# if 'first_tree' in locals():
#     visualize_tree(first_tree.root, 
#                    feature_names_list, 
#                    filename='C5.0_Tree_High_Res', 
#                    format='png',
#                    dpi=3000) # dpi=300 就很清楚了
# else:
#     print("請先執行上一個 Cell 來準備變數。")

使用網格找出最佳參數組合

In [ ]:
import time
from itertools import product

# --- 步驟 1: 定義您想搜尋的參數網格 ---

# Key 是參數名稱，Value 是您想嘗試的值的列表
param_grid = {
    'max_depth': [5,15,25],
    'min_samples_split': [50,250,550],
    'min_samples_leaf': [10,50,100,150],
    'n_estimators':[5,10,15,20,25]
}

# --- 步驟 2: 進行網格搜尋 ---

# 取得所有參數的名稱和值
param_names = list(param_grid.keys())
param_values = list(param_grid.values())

# 使用 itertools.product 產生所有可能的參數組合
all_param_combinations = list(product(*param_values))

print(f"網格搜尋開始！總共要測試 {len(all_param_combinations)} 種參數組合。")
print("-" * 50)

# 用來記錄最佳結果的變數
best_accuracy = 0.0
best_params = {}
best_model = None

# --- 步驟 3: 迭代所有組合，訓練並評估模型 ---

start_time = time.time()

for i, params_tuple in enumerate(all_param_combinations):
    
    # 將參數組合打包成一個字典
    current_params = dict(zip(param_names, params_tuple))
    
    # 分離出 C50Boosted 和 C50DecisionTree 的參數
    n_estimators_val = current_params.pop('n_estimators')
    tree_params_val = current_params
    
    #print(f"({i+1}/{len(all_param_combinations)}) 測試中: {tree_params_val}, n_estimators={n_estimators_val}")
    print(f"--- 正在測試第 {i+1} / {len(all_param_combinations)} 個組合 ---")
    
    # 建立、訓練模型
    model_gs = C50Boosted(n_estimators=n_estimators_val, **tree_params_val)
    
    # 注意：這裡的 fit 方法內部有 print 輸出，在搜尋時可能會產生大量訊息。
    # 如果想保持乾淨，可以暫時修改 C50Boosted 的 fit 函式，將 print 註解掉。
    model_gs.fit(
        X_train, y_train,
        categorical_features=preprocessor.categorical_features,
        numerical_features=preprocessor.numerical_features
    )
    
    # 在「測試集」上進行預測與評估
    y_pred_gs = model_gs.predict(X_test)
    metrics = evaluate_model(y_test, y_pred_gs)
    accuracy = metrics['accuracy']
    
    print(f"  -> 測試集準確率: {accuracy * 100:.2f}%")
    
    # 如果目前的準確率比紀錄中的最佳準確率還高，就更新紀錄
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_params = {**tree_params_val, 'n_estimators': n_estimators_val} # 存回完整參數
        best_model = model_gs # 也可以把最好的模型存起來
        print(f"  ✨ 新的最佳準確率！")

end_time = time.time()

# --- 步驟 4: 顯示最終結果 ---
print("\n" + "=" * 50)
print("網格搜尋完成！")
print(f"執行時間: {end_time - start_time:.2f} 秒")
print("=" * 50)
print(f"最高準確率: {best_accuracy * 100:.2f}%")
print(f"最佳參數配置: {best_params}")